# Amazon Review Data 2018 — Week 1 V4 EDA (Visual Version)

This version keeps the V4 tables **and adds more visual EDA plots** so the notebook is easier to present to a leader.

### Upload
Required:
- `amazon_week1_v4_processed_data.zip`

Optional:
- `manual_validation_50_completed.csv`

If the manual-validation CSV is uploaded together with the V4 ZIP, the notebook will also visualize the 50-query human-validation results.


In [ ]:

# ============================================================
# Amazon Review Data 2018 — Week 1 V4 EDA One-Click Colab
#
# Input:
#   amazon_week1_v4_processed_data.zip
#   OR the V4 parquet files directly
#
# Output:
#   - V4-specific EDA tables
#   - relation-graph coverage analysis
#   - relevance-label / relation-type analysis
#   - candidate coverage per query
#   - query quality analysis
#   - missing-value analysis
#   - cold-start / low-history analysis
#   - sanity-check samples
#   - Markdown report
#   - ZIP of all results
#
# IMPORTANT:
# V4 uses Amazon relation signals (also_buy / also_view) for
# positive relevance labels. TF-IDF similarity is diagnostic /
# hard-negative mining only, not the source of positive labels.
# ============================================================

!pip -q install pandas pyarrow matplotlib

import os
import re
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from IPython.display import display, Markdown


# ============================================================
# 1. UPLOAD V4 OUTPUT
# ============================================================

print("Upload ONE of the following:")
print("1) amazon_week1_v4_processed_data.zip")
print("OR")
print("2) cleaned_products_10k.parquet")
print("   normalized_queries.parquet")
print("   query_product_relevance.parquet")

uploaded = files.upload()

BASE_DIR = Path("/content/amazon_week1_v4_eda")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
FIG_DIR = OUTPUT_DIR / "figures"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

for name, data in uploaded.items():
    path = INPUT_DIR / name
    with open(path, "wb") as f:
        f.write(data)

# Unzip if needed
for zip_path in INPUT_DIR.glob("*.zip"):
    print("Extracting:", zip_path.name)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(INPUT_DIR)


# ============================================================
# 2. LOCATE FILES
# ============================================================

def find_file(filename):
    matches = list(INPUT_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. "
            "Make sure you uploaded the V4 processed ZIP/files."
        )

    return matches[0]


products_path = find_file(
    "cleaned_products_10k.parquet"
)

queries_path = find_file(
    "normalized_queries.parquet"
)

pairs_path = find_file(
    "query_product_relevance.parquet"
)


products = pd.read_parquet(
    products_path
)

queries = pd.read_parquet(
    queries_path
)

pairs = pd.read_parquet(
    pairs_path
)


print("\nLoaded V4 data:")
print("products:", products.shape)
print("queries :", queries.shape)
print("pairs   :", pairs.shape)

print("\nPair columns:")
print(list(pairs.columns))


# ============================================================
# 3. VERIFY THIS IS V4
# ============================================================

required_v4_pair_columns = {
    "label_relation",
    "label_source",
    "tfidf_similarity_diagnostic"
}

missing_v4_cols = (
    required_v4_pair_columns
    - set(pairs.columns)
)

if missing_v4_cols:
    raise ValueError(
        "This does not look like V4 output. "
        f"Missing V4 columns: {missing_v4_cols}"
    )

if "label_source" in pairs.columns:

    label_sources = (
        pairs["label_source"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    print("\nDetected label_source:")
    print(label_sources)


# ============================================================
# 4. BASIC QUALITY CHECKS
# ============================================================

quality_rows = [
    [
        "Products",
        "Rows",
        len(products)
    ],
    [
        "Products",
        "Unique product_id",
        products["product_id"].nunique()
    ],
    [
        "Products",
        "Duplicate product_id",
        products["product_id"].duplicated().sum()
    ],
    [
        "Queries",
        "Rows",
        len(queries)
    ],
    [
        "Queries",
        "Unique query_id",
        queries["query_id"].nunique()
    ],
    [
        "Queries",
        "Duplicate query text",
        queries["query"].duplicated().sum()
    ],
    [
        "Pairs",
        "Rows",
        len(pairs)
    ],
    [
        "Pairs",
        "Unique query_id",
        pairs["query_id"].nunique()
    ],
    [
        "Pairs",
        "Duplicate query-product pair",
        pairs.duplicated(
            ["query_id", "product_id"]
        ).sum()
    ],
]

quality_report = pd.DataFrame(
    quality_rows,
    columns=[
        "dataset",
        "check",
        "value"
    ]
)

display(
    Markdown(
        "## 1. Basic Quality Checks"
    )
)

display(
    quality_report
)

quality_report.to_csv(
    OUTPUT_DIR / "01_quality_checks.csv",
    index=False
)


# ============================================================
# 5. AMAZON RELATION GRAPH COVERAGE
# ============================================================

display(
    Markdown(
        "## 2. Amazon Relation Graph Coverage"
    )
)

relation_metrics = []

def safe_len(x):
    return len(x) if isinstance(x, list) else 0


if "also_buy_in_sample" in products.columns:

    buy_counts = products[
        "also_buy_in_sample"
    ].apply(safe_len)

    relation_metrics.append([
        "Products with >=1 also_buy relation",
        int((buy_counts > 0).sum()),
        float((buy_counts > 0).mean())
    ])

    relation_metrics.append([
        "Mean also_buy relations per product",
        float(buy_counts.mean()),
        np.nan
    ])


if "also_view_in_sample" in products.columns:

    view_counts = products[
        "also_view_in_sample"
    ].apply(safe_len)

    relation_metrics.append([
        "Products with >=1 also_view relation",
        int((view_counts > 0).sum()),
        float((view_counts > 0).mean())
    ])

    relation_metrics.append([
        "Mean also_view relations per product",
        float(view_counts.mean()),
        np.nan
    ])


if "direct_relation_count_in_sample" in products.columns:

    direct_counts = pd.to_numeric(
        products[
            "direct_relation_count_in_sample"
        ],
        errors="coerce"
    ).fillna(0)

    relation_metrics.append([
        "Products with >=1 direct relation",
        int((direct_counts > 0).sum()),
        float((direct_counts > 0).mean())
    ])

    relation_metrics.append([
        "Mean direct relations per product",
        float(direct_counts.mean()),
        np.nan
    ])

    relation_metrics.append([
        "Median direct relations per product",
        float(direct_counts.median()),
        np.nan
    ])

relation_summary = pd.DataFrame(
    relation_metrics,
    columns=[
        "metric",
        "value",
        "rate"
    ]
)

display(
    relation_summary
)

relation_summary.to_csv(
    OUTPUT_DIR / "02_relation_graph_summary.csv",
    index=False
)


if "direct_relation_count_in_sample" in products.columns:

    plt.figure(
        figsize=(9, 5)
    )

    capped = np.minimum(
        direct_counts,
        direct_counts.quantile(0.99)
    )

    plt.hist(
        capped,
        bins=30
    )

    plt.title(
        "Direct Amazon Relations per Product (capped at 99th percentile)"
    )

    plt.xlabel(
        "Direct relation count"
    )

    plt.ylabel(
        "Product count"
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "01_direct_relation_count_distribution.png",
        dpi=150
    )

    plt.show()


# ============================================================
# 6. QUERY ANALYSIS
# ============================================================

display(
    Markdown(
        "## 3. Query Analysis"
    )
)

queries["query_length_tokens"] = (
    queries["query"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

query_length_stats = (
    queries[
        "query_length_tokens"
    ]
    .describe()
    .reset_index()
)

query_length_stats.columns = [
    "statistic",
    "value"
]

display(
    Markdown(
        "### Query Length Statistics"
    )
)

display(
    query_length_stats
)

query_length_stats.to_csv(
    OUTPUT_DIR / "03_query_length_statistics.csv",
    index=False
)


plt.figure(
    figsize=(9, 5)
)

plt.hist(
    queries[
        "query_length_tokens"
    ],
    bins=20
)

plt.title(
    "Pseudo Query Length Distribution"
)

plt.xlabel(
    "Number of tokens"
)

plt.ylabel(
    "Query count"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "02_query_length_distribution.png",
    dpi=150
)

plt.show()


if "query_source" in queries.columns:

    query_source_dist = (
        queries[
            "query_source"
        ]
        .fillna("unknown")
        .value_counts()
        .reset_index()
    )

    query_source_dist.columns = [
        "query_source",
        "count"
    ]

    query_source_dist[
        "rate"
    ] = (
        query_source_dist[
            "count"
        ]
        / query_source_dist[
            "count"
        ].sum()
    )

    display(
        Markdown(
            "### Query Source Distribution"
        )
    )

    display(
        query_source_dist
    )

    query_source_dist.to_csv(
        OUTPUT_DIR / "04_query_source_distribution.csv",
        index=False
    )


# ============================================================
# 7. RELEVANCE LABEL DISTRIBUTION
# ============================================================

display(
    Markdown(
        "## 4. Relevance Label Distribution"
    )
)

label_dist = (
    pairs[
        "relevance_label"
    ]
    .value_counts()
    .sort_index()
    .reset_index()
)

label_dist.columns = [
    "relevance_label",
    "count"
]

label_dist[
    "rate"
] = (
    label_dist[
        "count"
    ]
    / label_dist[
        "count"
    ].sum()
)

display(
    label_dist
)

label_dist.to_csv(
    OUTPUT_DIR / "05_relevance_label_distribution.csv",
    index=False
)


plt.figure(
    figsize=(8, 5)
)

plt.bar(
    label_dist[
        "relevance_label"
    ].astype(str),
    label_dist[
        "count"
    ]
)

plt.title(
    "V4 Relevance Label Distribution"
)

plt.xlabel(
    "Relevance label"
)

plt.ylabel(
    "Query-product pair count"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "03_relevance_label_distribution.png",
    dpi=150
)

plt.show()


# ============================================================
# 8. LABEL RELATION TYPES
# ============================================================

display(
    Markdown(
        "## 5. V4 Label Relation Types"
    )
)

relation_dist = (
    pairs[
        "label_relation"
    ]
    .fillna("unknown")
    .value_counts()
    .reset_index()
)

relation_dist.columns = [
    "label_relation",
    "count"
]

relation_dist[
    "rate"
] = (
    relation_dist[
        "count"
    ]
    / relation_dist[
        "count"
    ].sum()
)

display(
    relation_dist
)

relation_dist.to_csv(
    OUTPUT_DIR / "06_label_relation_distribution.csv",
    index=False
)


plt.figure(
    figsize=(10, 6)
)

plt.barh(
    relation_dist[
        "label_relation"
    ][::-1],
    relation_dist[
        "count"
    ][::-1]
)

plt.title(
    "V4 Label Relation Type Distribution"
)

plt.xlabel(
    "Pair count"
)

plt.ylabel(
    "Relation type"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "04_label_relation_distribution.png",
    dpi=150
)

plt.show()


# ============================================================
# 9. CANDIDATE COVERAGE PER QUERY
# ============================================================

display(
    Markdown(
        "## 6. Candidate Coverage per Query"
    )
)

per_query = (
    pairs.groupby(
        "query_id"
    )
    .agg(
        candidate_count=(
            "product_id",
            "count"
        ),
        label3_count=(
            "relevance_label",
            lambda s: int(
                (s == 3).sum()
            )
        ),
        label2_count=(
            "relevance_label",
            lambda s: int(
                (s == 2).sum()
            )
        ),
        label1_count=(
            "relevance_label",
            lambda s: int(
                (s == 1).sum()
            )
        ),
        label0_count=(
            "relevance_label",
            lambda s: int(
                (s == 0).sum()
            )
        )
    )
    .reset_index()
)

per_query[
    "has_direct_positive"
] = (
    per_query[
        "label2_count"
    ] > 0
)

per_query[
    "has_weak_positive"
] = (
    per_query[
        "label1_count"
    ] > 0
)

coverage_summary = pd.DataFrame({
    "metric": [
        "Mean candidates/query",
        "Median candidates/query",
        "Queries with label-2 direct positive",
        "Rate with label-2 direct positive",
        "Queries with label-1 weak positive",
        "Rate with label-1 weak positive",
    ],
    "value": [
        float(
            per_query[
                "candidate_count"
            ].mean()
        ),
        float(
            per_query[
                "candidate_count"
            ].median()
        ),
        int(
            per_query[
                "has_direct_positive"
            ].sum()
        ),
        float(
            per_query[
                "has_direct_positive"
            ].mean()
        ),
        int(
            per_query[
                "has_weak_positive"
            ].sum()
        ),
        float(
            per_query[
                "has_weak_positive"
            ].mean()
        ),
    ]
})

display(
    coverage_summary
)

display(
    Markdown(
        "### Candidate Count Statistics"
    )
)

display(
    per_query[
        "candidate_count"
    ].describe()
)

coverage_summary.to_csv(
    OUTPUT_DIR / "07_query_candidate_coverage_summary.csv",
    index=False
)

per_query.to_csv(
    OUTPUT_DIR / "08_query_candidate_coverage_detail.csv",
    index=False
)


plt.figure(
    figsize=(8, 5)
)

plt.hist(
    per_query[
        "candidate_count"
    ],
    bins=range(
        int(
            per_query[
                "candidate_count"
            ].min()
        ),
        int(
            per_query[
                "candidate_count"
            ].max()
        ) + 2
    ),
    align="left"
)

plt.title(
    "Candidate Products per Query"
)

plt.xlabel(
    "Candidate count"
)

plt.ylabel(
    "Query count"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "05_candidate_count_per_query.png",
    dpi=150
)

plt.show()


# ============================================================
# 10. LABEL-2 DIRECT RELATION ANALYSIS
# ============================================================

display(
    Markdown(
        "## 7. Direct Amazon Positive Analysis"
    )
)

direct_positive_pairs = pairs[
    pairs[
        "relevance_label"
    ] == 2
].copy()

if len(
    direct_positive_pairs
) > 0:

    direct_relation_type = (
        direct_positive_pairs[
            "label_relation"
        ]
        .value_counts()
        .reset_index()
    )

    direct_relation_type.columns = [
        "direct_relation_type",
        "count"
    ]

    direct_relation_type[
        "rate"
    ] = (
        direct_relation_type[
            "count"
        ]
        / direct_relation_type[
            "count"
        ].sum()
    )

    display(
        direct_relation_type
    )

    direct_relation_type.to_csv(
        OUTPUT_DIR / "09_direct_positive_relation_types.csv",
        index=False
    )

else:

    print(
        "No label-2 direct positive pairs found."
    )


# ============================================================
# 11. TF-IDF DIAGNOSTIC ANALYSIS
# ============================================================
#
# This is diagnostic only. In V4 TF-IDF did not create
# positive labels; it was used for hard-negative mining.
# ============================================================

display(
    Markdown(
        "## 8. TF-IDF Similarity Diagnostic by Label"
    )
)

if "tfidf_similarity_diagnostic" in pairs.columns:

    tfidf_by_label = (
        pairs.groupby(
            "relevance_label"
        )[
            "tfidf_similarity_diagnostic"
        ]
        .agg(
            [
                "count",
                "mean",
                "median",
                "min",
                "max"
            ]
        )
        .reset_index()
    )

    display(
        tfidf_by_label
    )

    tfidf_by_label.to_csv(
        OUTPUT_DIR / "10_tfidf_diagnostic_by_label.csv",
        index=False
    )

    plt.figure(
        figsize=(9, 5)
    )

    for label in sorted(
        pairs[
            "relevance_label"
        ].dropna().unique()
    ):

        vals = pairs.loc[
            pairs[
                "relevance_label"
            ] == label,
            "tfidf_similarity_diagnostic"
        ].dropna()

        if len(vals):
            plt.hist(
                vals,
                bins=30,
                alpha=0.4,
                label=f"label {int(label)}"
            )

    plt.title(
        "TF-IDF Similarity by V4 Relevance Label"
    )

    plt.xlabel(
        "TF-IDF similarity"
    )

    plt.ylabel(
        "Pair count"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "06_tfidf_similarity_by_label.png",
        dpi=150
    )

    plt.show()


# ============================================================
# 12. MISSING VALUE ANALYSIS
# ============================================================

display(
    Markdown(
        "## 9. Missing Value Analysis"
    )
)

fields_to_check = [
    c
    for c in [
        "title",
        "description",
        "image_url",
        "brand",
        "price",
        "average_rating",
        "review_count"
    ]
    if c in products.columns
]

missing_rows = []

for col in fields_to_check:

    if col in [
        "title",
        "description",
        "image_url",
        "brand"
    ]:

        missing_mask = (
            products[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
        )

    else:

        missing_mask = (
            products[col]
            .isna()
        )

    missing_rows.append({
        "field": col,
        "missing_count": int(
            missing_mask.sum()
        ),
        "missing_rate": float(
            missing_mask.mean()
        )
    })

missing_report = (
    pd.DataFrame(
        missing_rows
    )
    .sort_values(
        "missing_rate",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

display(
    missing_report
)

missing_report.to_csv(
    OUTPUT_DIR / "11_missing_value_summary.csv",
    index=False
)


plt.figure(
    figsize=(9, 5)
)

plt.bar(
    missing_report[
        "field"
    ],
    missing_report[
        "missing_rate"
    ]
)

plt.title(
    "Missing Value Rate by Product Field"
)

plt.xlabel(
    "Field"
)

plt.ylabel(
    "Missing rate"
)

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "07_missing_value_rate.png",
    dpi=150
)

plt.show()


# ============================================================
# 13. PRICE / RATING / REVIEW COUNT
# ============================================================

display(
    Markdown(
        "## 10. Product Statistics"
    )
)

if "price" in products.columns:

    price = pd.to_numeric(
        products[
            "price"
        ],
        errors="coerce"
    ).dropna()

    price = price[
        price > 0
    ]

    if len(
        price
    ):

        price_stats = (
            price.describe(
                percentiles=[
                    0.25,
                    0.5,
                    0.75,
                    0.9,
                    0.99
                ]
            )
            .reset_index()
        )

        price_stats.columns = [
            "statistic",
            "value"
        ]

        display(
            Markdown(
                "### Price Statistics"
            )
        )

        display(
            price_stats
        )

        price_stats.to_csv(
            OUTPUT_DIR / "12_price_statistics.csv",
            index=False
        )

        upper = price.quantile(
            0.99
        )

        plt.figure(
            figsize=(9, 5)
        )

        plt.hist(
            price[
                price <= upper
            ],
            bins=50
        )

        plt.title(
            "Product Price Distribution (<= 99th percentile)"
        )

        plt.xlabel(
            "Price"
        )

        plt.ylabel(
            "Product count"
        )

        plt.tight_layout()

        plt.savefig(
            FIG_DIR / "08_price_distribution.png",
            dpi=150
        )

        plt.show()


if "average_rating" in products.columns:

    rating = pd.to_numeric(
        products[
            "average_rating"
        ],
        errors="coerce"
    ).dropna()

    plt.figure(
        figsize=(9, 5)
    )

    plt.hist(
        rating,
        bins=20
    )

    plt.title(
        "Average Product Rating Distribution"
    )

    plt.xlabel(
        "Average rating"
    )

    plt.ylabel(
        "Product count"
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "09_rating_distribution.png",
        dpi=150
    )

    plt.show()


if "review_count" in products.columns:

    rc = pd.to_numeric(
        products[
            "review_count"
        ],
        errors="coerce"
    ).fillna(0)

    plt.figure(
        figsize=(9, 5)
    )

    plt.hist(
        np.log1p(rc),
        bins=40
    )

    plt.title(
        "Review Count Distribution (log1p scale)"
    )

    plt.xlabel(
        "log(1 + review_count)"
    )

    plt.ylabel(
        "Product count"
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR / "10_review_count_distribution.png",
        dpi=150
    )

    plt.show()


# ============================================================
# 14. COLD-START / LOW-HISTORY ANALYSIS
# ============================================================

display(
    Markdown(
        "## 11. Cold-Start / Low-History Analysis"
    )
)

if "review_count" in products.columns:

    rc = pd.to_numeric(
        products[
            "review_count"
        ],
        errors="coerce"
    ).fillna(0)

    products[
        "cold_start_strict"
    ] = rc.eq(0)

    products[
        "low_history_1"
    ] = rc.le(1)

    products[
        "low_history_3"
    ] = rc.le(3)

    cold_summary = pd.DataFrame({
        "definition": [
            "strict: review_count == 0",
            "low-history: review_count <= 1",
            "low-history: review_count <= 3"
        ],
        "product_count": [
            int(
                products[
                    "cold_start_strict"
                ].sum()
            ),
            int(
                products[
                    "low_history_1"
                ].sum()
            ),
            int(
                products[
                    "low_history_3"
                ].sum()
            )
        ],
        "rate": [
            float(
                products[
                    "cold_start_strict"
                ].mean()
            ),
            float(
                products[
                    "low_history_1"
                ].mean()
            ),
            float(
                products[
                    "low_history_3"
                ].mean()
            )
        ]
    })

    display(
        cold_summary
    )

    cold_summary.to_csv(
        OUTPUT_DIR / "13_cold_start_summary.csv",
        index=False
    )


# ============================================================
# 15. SANITY CHECK — RANDOM QUERIES
# ============================================================

display(
    Markdown(
        "## 12. Sanity Check — 15 Random Queries"
    )
)

sample_qids = (
    pairs[
        "query_id"
    ]
    .drop_duplicates()
    .sample(
        n=min(
            15,
            pairs[
                "query_id"
            ].nunique()
        ),
        random_state=42
    )
    .tolist()
)

sanity = (
    pairs[
        pairs[
            "query_id"
        ].isin(
            sample_qids
        )
    ]
    .sort_values(
        [
            "query_id",
            "relevance_label",
            "label_relation"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    [
        [
            c
            for c in [
                "query_id",
                "query",
                "relevance_label",
                "label_relation",
                "product_id",
                "title",
                "brand",
                "category",
                "tfidf_similarity_diagnostic"
            ]
            if c in pairs.columns
        ]
    ]
)

display(
    sanity
)

sanity.to_csv(
    OUTPUT_DIR / "14_sanity_check_15_queries.csv",
    index=False
)


# ============================================================
# 16. RELATION-SPECIFIC SANITY SAMPLES
# ============================================================

display(
    Markdown(
        "## 13. Direct Relation Positive Samples"
    )
)

direct_sample = (
    pairs[
        pairs[
            "relevance_label"
        ] == 2
    ]
    [
        [
            c
            for c in [
                "query_id",
                "query",
                "label_relation",
                "product_id",
                "title",
                "brand",
                "category"
            ]
            if c in pairs.columns
        ]
    ]
)

if len(
    direct_sample
):

    direct_sample = direct_sample.sample(
        n=min(
            30,
            len(
                direct_sample
            )
        ),
        random_state=42
    )

    display(
        direct_sample
    )

    direct_sample.to_csv(
        OUTPUT_DIR / "15_direct_positive_samples.csv",
        index=False
    )

else:

    print(
        "No direct-positive samples available."
    )


# ============================================================
# 17. QUERY SOURCE × POSITIVE COVERAGE
# ============================================================

if "query_source" in queries.columns:

    coverage_with_query_source = per_query.merge(
        queries[
            [
                "query_id",
                "query_source"
            ]
        ],
        on="query_id",
        how="left"
    )

    query_source_coverage = (
        coverage_with_query_source
        .groupby(
            "query_source"
        )
        .agg(
            query_count=(
                "query_id",
                "count"
            ),
            mean_candidate_count=(
                "candidate_count",
                "mean"
            ),
            direct_positive_rate=(
                "has_direct_positive",
                "mean"
            ),
            weak_positive_rate=(
                "has_weak_positive",
                "mean"
            )
        )
        .reset_index()
    )

    display(
        Markdown(
            "## 14. Query Source × Positive Coverage"
        )
    )

    display(
        query_source_coverage
    )

    query_source_coverage.to_csv(
        OUTPUT_DIR / "16_query_source_positive_coverage.csv",
        index=False
    )


# ============================================================
# 18. DATASET SCHEMA
# ============================================================

schema_rows = []

for col in pairs.columns:
    schema_rows.append({
        "field": col,
        "dtype": str(
            pairs[col].dtype
        ),
        "description": ""
    })

description_map = {
    "query_id": "Unique pseudo-query identifier",
    "query": "Normalized pseudo search query",
    "product_id": "Amazon ASIN",
    "title": "Cleaned product title",
    "description": "Cleaned product description",
    "image_url": "Primary product image URL",
    "brand": "Product brand",
    "price": "Parsed numeric price",
    "average_rating": "Average rating aggregated from reviews",
    "review_count": "Number of reviews for the product",
    "relevance_label": "V4 heuristic relevance label: 0/1/2/3",
    "label_relation": "Reason a query-product pair received its V4 label",
    "label_source": "V4 relation-graph heuristic label source",
    "tfidf_similarity_diagnostic": "TF-IDF diagnostic / hard-negative-mining similarity, not positive-label source"
}

schema = pd.DataFrame(
    schema_rows
)

schema[
    "description"
] = schema[
    "field"
].map(
    description_map
).fillna("")

display(
    Markdown(
        "## 15. Dataset Schema"
    )
)

display(
    schema
)

schema.to_csv(
    OUTPUT_DIR / "17_dataset_schema.csv",
    index=False
)


# ============================================================
# 19. BUILD AUTOMATIC EDA REPORT
# ============================================================

def get_label_count(label):
    row = label_dist[
        label_dist[
            "relevance_label"
        ] == label
    ]

    return (
        int(
            row[
                "count"
            ].iloc[0]
        )
        if len(
            row
        )
        else 0
    )


direct_positive_rate = float(
    per_query[
        "has_direct_positive"
    ].mean()
)

weak_positive_rate = float(
    per_query[
        "has_weak_positive"
    ].mean()
)

strict_cold_count = (
    int(
        products[
            "cold_start_strict"
        ].sum()
    )
    if "cold_start_strict" in products.columns
    else 0
)

low_history_count = (
    int(
        products[
            "low_history_1"
        ].sum()
    )
    if "low_history_1" in products.columns
    else 0
)


report = f"""# Amazon Review Data 2018 — Week 1 V4 EDA Report

## 1. Dataset Overview
- Products: {len(products):,}
- Pseudo queries: {len(queries):,}
- Query-product pairs: {len(pairs):,}
- Unique products: {products['product_id'].nunique():,}
- Unique queries: {queries['query_id'].nunique():,}
- Duplicate product IDs: {products['product_id'].duplicated().sum():,}
- Duplicate query-product pairs: {pairs.duplicated(['query_id','product_id']).sum():,}

## 2. V4 Labeling Strategy
V4 no longer uses TF-IDF similarity to create positive relevance labels.

- Label 3: source product for the pseudo query
- Label 2: direct Amazon product relation (`also_buy` / `also_view`)
- Label 1: second-hop graph relation or same-brand weak relation
- Label 0: lexical hard negative or random negative

TF-IDF similarity is retained only as a diagnostic and for hard-negative mining.

## 3. Label Distribution
- Label 3 pairs: {get_label_count(3):,}
- Label 2 pairs: {get_label_count(2):,}
- Label 1 pairs: {get_label_count(1):,}
- Label 0 pairs: {get_label_count(0):,}

## 4. Positive-Candidate Coverage
- Mean candidates per query: {per_query['candidate_count'].mean():.2f}
- Median candidates per query: {per_query['candidate_count'].median():.2f}
- Queries with at least one direct Amazon positive (label 2): {direct_positive_rate:.1%}
- Queries with at least one weak positive (label 1): {weak_positive_rate:.1%}

## 5. Query Characteristics
- Mean query length: {queries['query_length_tokens'].mean():.2f} tokens
- Median query length: {queries['query_length_tokens'].median():.2f} tokens

Queries remain pseudo queries generated from filtered review summaries or cleaned product titles.

## 6. Metadata Missingness
See `11_missing_value_summary.csv` and the missing-value figure for field-level sparsity.

Missing optional metadata is retained instead of dropping large portions of the catalog.

## 7. Cold-Start / Low-History Analysis
- Strict zero-review products: {strict_cold_count:,}
- Low-history products (`review_count <= 1`): {low_history_count:,}

Because Amazon Review Data 2018 is itself review-based, strict zero-history items can be rare. Review count is only a proxy for interaction history.

## 8. Main Limitation
Amazon Review Data 2018 does not provide real search queries, search-click logs, or explicit query-product relevance judgments.

Therefore the V4 dataset is still a heuristic ranking dataset:
- pseudo queries are generated from review summaries or titles
- positive labels are based on Amazon-provided product-relation signals
- negative labels are mined using graph exclusion plus lexical/random negative sampling

This is stronger than the earlier lexical-only pseudo-label approach, but it should not be described as ground-truth Amazon search relevance.

## 9. Week 1 Readiness
If the sanity-check samples show that label-2 direct relations are meaningfully related to the pseudo queries, the V4 dataset is ready for Week 2 baseline ranking experiments:
- BM25
- TF-IDF
- XGBoost pointwise ranking
- NDCG@10
- MAP@10
- MRR
"""

with open(
    OUTPUT_DIR / "V4_EDA_Report.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        report
    )

display(
    Markdown(
        report
    )
)


# ============================================================
# 20. SAVE ENRICHED DATA
# ============================================================

products.to_parquet(
    OUTPUT_DIR / "cleaned_products_10k_with_eda_flags.parquet",
    index=False
)

queries.to_parquet(
    OUTPUT_DIR / "normalized_queries_with_length.parquet",
    index=False
)


# ============================================================
# 21. ZIP + DOWNLOAD
# ============================================================

zip_path = shutil.make_archive(
    "/content/amazon_week1_v4_eda_results",
    "zip",
    root_dir=OUTPUT_DIR
)

print("\n==========================================")
print("V4 EDA COMPLETE")
print("Results ZIP:", zip_path)
print("==========================================")

files.download(
    zip_path
)


# ============================================================
# 22. EXTRA VISUAL EDA — PRESENTATION-FRIENDLY PLOTS
# ============================================================

display(Markdown("## 16. Extra Visual EDA"))

# ------------------------------------------------------------
# A. Query source distribution
# ------------------------------------------------------------
if "query_source" in queries.columns:
    qs = (
        queries["query_source"]
        .fillna("unknown")
        .value_counts()
        .sort_values(ascending=True)
    )

    plt.figure(figsize=(8, 5))
    plt.barh(qs.index, qs.values)
    plt.title("Query Source Distribution")
    plt.xlabel("Number of queries")
    plt.ylabel("Query source")
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "11_query_source_distribution.png",
        dpi=150
    )
    plt.show()


# ------------------------------------------------------------
# B. Positive coverage rates
# ------------------------------------------------------------
coverage_plot = pd.DataFrame({
    "metric": [
        "Queries with direct positive (Label 2)",
        "Queries with weak positive (Label 1)"
    ],
    "rate": [
        float(per_query["has_direct_positive"].mean()),
        float(per_query["has_weak_positive"].mean())
    ]
})

plt.figure(figsize=(9, 5))
plt.bar(
    coverage_plot["metric"],
    coverage_plot["rate"]
)
plt.title("Positive Candidate Coverage")
plt.ylabel("Rate")
plt.ylim(0, 1.05)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(
    FIG_DIR / "12_positive_candidate_coverage.png",
    dpi=150
)
plt.show()


# ------------------------------------------------------------
# C. Direct Amazon relation types
# ------------------------------------------------------------
if len(direct_positive_pairs) > 0:
    direct_type_plot = (
        direct_positive_pairs["label_relation"]
        .value_counts()
        .sort_values(ascending=True)
    )

    plt.figure(figsize=(8, 5))
    plt.barh(
        direct_type_plot.index,
        direct_type_plot.values
    )
    plt.title("Label-2 Direct Amazon Relation Types")
    plt.xlabel("Query-product pairs")
    plt.ylabel("Relation type")
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "13_direct_relation_types.png",
        dpi=150
    )
    plt.show()


# ------------------------------------------------------------
# D. Candidate count boxplot
# ------------------------------------------------------------
plt.figure(figsize=(7, 5))
plt.boxplot(
    per_query["candidate_count"].dropna(),
    vert=True
)
plt.title("Candidate Count per Query")
plt.ylabel("Candidate count")
plt.xticks([1], ["All queries"])
plt.tight_layout()
plt.savefig(
    FIG_DIR / "14_candidate_count_boxplot.png",
    dpi=150
)
plt.show()


# ------------------------------------------------------------
# E. TF-IDF diagnostic boxplot by relevance label
# ------------------------------------------------------------
if "tfidf_similarity_diagnostic" in pairs.columns:
    labels_sorted = sorted(
        pairs["relevance_label"]
        .dropna()
        .unique()
    )

    box_data = [
        pairs.loc[
            pairs["relevance_label"] == label,
            "tfidf_similarity_diagnostic"
        ].dropna().values
        for label in labels_sorted
    ]

    plt.figure(figsize=(8, 5))
    plt.boxplot(
        box_data,
        labels=[
            str(int(x))
            for x in labels_sorted
        ],
        showfliers=False
    )
    plt.title("TF-IDF Similarity Diagnostic by Relevance Label")
    plt.xlabel("Relevance label")
    plt.ylabel("TF-IDF similarity")
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "15_tfidf_similarity_boxplot.png",
        dpi=150
    )
    plt.show()


# ------------------------------------------------------------
# F. Cold-start / low-history rates
# ------------------------------------------------------------
if "review_count" in products.columns:
    rc = pd.to_numeric(
        products["review_count"],
        errors="coerce"
    ).fillna(0)

    cold_plot = pd.DataFrame({
        "definition": [
            "review_count = 0",
            "review_count <= 1",
            "review_count <= 3"
        ],
        "rate": [
            float(rc.eq(0).mean()),
            float(rc.le(1).mean()),
            float(rc.le(3).mean())
        ]
    })

    plt.figure(figsize=(9, 5))
    plt.bar(
        cold_plot["definition"],
        cold_plot["rate"]
    )
    plt.title("Cold-Start / Low-History Product Rates")
    plt.ylabel("Share of products")
    plt.ylim(0, 1.0)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(
        FIG_DIR / "16_cold_start_rates.png",
        dpi=150
    )
    plt.show()


# ------------------------------------------------------------
# G. Relevance-label mix as percentages
# ------------------------------------------------------------
label_pct = (
    pairs["relevance_label"]
    .value_counts(normalize=True)
    .sort_index()
)

plt.figure(figsize=(8, 5))
plt.bar(
    label_pct.index.astype(str),
    label_pct.values
)
plt.title("Relevance Label Mix")
plt.xlabel("Relevance label")
plt.ylabel("Share of query-product pairs")
plt.ylim(0, max(label_pct.values) * 1.2)
plt.tight_layout()
plt.savefig(
    FIG_DIR / "17_relevance_label_mix_percentage.png",
    dpi=150
)
plt.show()


# ------------------------------------------------------------
# H. Optional: manual validation plots
# ------------------------------------------------------------
manual_files = list(INPUT_DIR.rglob("manual_validation_50_completed.csv"))

if manual_files:
    manual = pd.read_csv(manual_files[0])

    if "human_judgment" in manual.columns:
        display(Markdown("## 17. Manual Validation Visuals"))

        judgment_order = ["Yes", "Maybe", "No"]
        judgment_counts = (
            manual["human_judgment"]
            .value_counts()
            .reindex(judgment_order)
            .fillna(0)
        )

        plt.figure(figsize=(7, 5))
        plt.bar(
            judgment_counts.index,
            judgment_counts.values
        )
        plt.title("Manual Validation — 50 Label-2 Pairs")
        plt.xlabel("Human judgment")
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(
            FIG_DIR / "18_manual_validation_overall.png",
            dpi=150
        )
        plt.show()

        if "label_relation" in manual.columns:
            relation_judgment = pd.crosstab(
                manual["label_relation"],
                manual["human_judgment"]
            )

            for col in judgment_order:
                if col not in relation_judgment.columns:
                    relation_judgment[col] = 0

            relation_judgment = relation_judgment[
                judgment_order
            ]

            # Separate bar chart per judgment to keep plots simple.
            for judgment in judgment_order:
                plt.figure(figsize=(8, 5))
                plt.bar(
                    relation_judgment.index,
                    relation_judgment[judgment].values
                )
                plt.title(
                    f"Manual Validation by Relation Type — {judgment}"
                )
                plt.xlabel("Amazon relation type")
                plt.ylabel("Count")
                plt.xticks(rotation=15, ha="right")
                plt.tight_layout()
                plt.savefig(
                    FIG_DIR / f"19_manual_validation_{judgment.lower()}.png",
                    dpi=150
                )
                plt.show()

            acceptable = (
                manual.assign(
                    acceptable=manual[
                        "human_judgment"
                    ].isin(["Yes", "Maybe"])
                )
                .groupby("label_relation")[
                    "acceptable"
                ]
                .mean()
                .sort_values(ascending=True)
            )

            plt.figure(figsize=(8, 5))
            plt.barh(
                acceptable.index,
                acceptable.values
            )
            plt.title(
                "Manual Validation — Acceptable Relevance Rate"
            )
            plt.xlabel("Yes + Maybe rate")
            plt.ylabel("Relation type")
            plt.xlim(0, 1.05)
            plt.tight_layout()
            plt.savefig(
                FIG_DIR / "20_manual_validation_acceptable_rate.png",
                dpi=150
            )
            plt.show()

else:
    print(
        "\nOptional manual-validation plot skipped. "
        "Upload manual_validation_50_completed.csv together with the V4 ZIP "
        "if you want the human-validation plots included."
    )


# ============================================================
# 23. RE-ZIP RESULTS WITH EXTRA FIGURES
# ============================================================

visual_zip_path = shutil.make_archive(
    "/content/amazon_week1_v4_eda_visual_results",
    "zip",
    root_dir=OUTPUT_DIR
)

print("\n==========================================")
print("VISUAL V4 EDA COMPLETE")
print("Results ZIP:", visual_zip_path)
print("==========================================")

files.download(
    visual_zip_path
)
